In [1]:
import anndata as ad
import numpy as np
import pandas as pd
from sccnasim.xlib.xdata import load_10x_data, save_10x_data

In [2]:
sc_fn = "/groups/cgsd/xianjie/data/dataset/HCC-atlas/HCC03N/HCC03N.count.h5ad"
st_dir = "/groups/cgsd/xianjie/projects/cna-benchmark/HCC3N_600spot/cna_prediction/1r1n1t/vary_tumor_prop/500n500t/gen_data/matrix"
st_anno_fn = "/groups/cgsd/xianjie/projects/cna-benchmark/HCC3N_600spot/cna_prediction/1r1n1t/vary_tumor_prop/500n500t/gen_data/matrix/spot_anno.tsv"
out_dir = "/groups/cgsd/xianjie/projects/cna-benchmark/HCC3N_600spot/cna_prediction/1r1n1t/vary_tumor_prop/analysis/infercnv_hepatocyte-ref/as_ref/gen_data"

n_normal = 30
n_tumor = 270
n_epi = 700

## Load single-cell data

In [3]:
adata_sc = ad.read_h5ad(sc_fn)
adata_sc

/home/xianjie/.anaconda3/envs/SCSC/lib/python3.11/site-packages/anndata-0.10.7-py3.11.egg/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


AnnData object with n_obs × n_vars = 2601 × 25712
    obs: 'cell', 'cell_type'
    var: 'gene'

In [4]:
adata_sc.obs['cell_type'].value_counts()

cell_type
Myeloid        1000
Hepatocyte      827
T/NK            414
Endothelial     218
B               121
Fibroblast       21
Name: count, dtype: int64

In [5]:
adata_sc.obs

,cell,cell_type
0,HCC03N_AAACCTGCAGTGAGTG,Myeloid
1,HCC03N_AAACCTGGTAAACGCG,Endothelial
2,HCC03N_AAACGGGAGATAGGAG,Hepatocyte
3,HCC03N_AAACGGGAGGAATCGC,Myeloid
4,HCC03N_AAACGGGAGGGTTTCT,Hepatocyte
...,...,...
2596,HCC03N_TTTGGTTGTCACAAGG,Myeloid
2597,HCC03N_TTTGGTTTCAGCCTAA,Myeloid
2598,HCC03N_TTTGTCAAGCCGGTAA,Hepatocyte
2599,HCC03N_TTTGTCACATTCTCAT,Hepatocyte


## Load ST data

In [6]:
adata_st = load_10x_data(st_dir)
adata_st

/home/xianjie/.anaconda3/envs/SCSC/lib/python3.11/site-packages/anndata-0.10.7-py3.11.egg/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/home/xianjie/.anaconda3/envs/SCSC/lib/python3.11/site-packages/anndata-0.10.7-py3.11.egg/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


AnnData object with n_obs × n_vars = 1600 × 33538
    obs: 'cell'
    var: 'feature_id', 'feature_name'

In [7]:
adata_st.X = adata_st.X.toarray()

In [8]:
anno = pd.read_csv(st_anno_fn, sep = '\t', header = None)
anno.columns = ['cell', 'cell_type']
anno

,cell,cell_type
0,AAACCTGCAAGTAGTA-1,ref
1,AAACCTGGTGTGAATA-1,ref
2,AAACCTGGTTTGCATG-1,ref
3,AAACCTGTCAGCGATT-1,normal
4,AAACCTGTCCTTTCTC-1,normal
...,...,...
1595,TTTGGTTGTAGCTTGT-1,tumor
1596,TTTGGTTGTGCTCTTC-1,tumor
1597,TTTGTCAAGGCTAGAC-1,tumor
1598,TTTGTCAGTGGGTCAA-1,tumor


In [9]:
adata_st.obs = adata_st.obs.merge(anno, on = 'cell', how = 'left')
adata_st.obs

,cell,cell_type
0,AAACCTGCAAGTAGTA-1,ref
1,AAACCTGGTGTGAATA-1,ref
2,AAACCTGGTTTGCATG-1,ref
3,AAACCTGTCAGCGATT-1,normal
4,AAACCTGTCCTTTCTC-1,normal
...,...,...
1595,TTTGGTTGTAGCTTGT-1,tumor
1596,TTTGGTTGTGCTCTTC-1,tumor
1597,TTTGTCAAGGCTAGAC-1,tumor
1598,TTTGTCAGTGGGTCAA-1,tumor


## Format Genes

In [10]:
overlap_genes = adata_sc.var['gene'][adata_sc.var['gene'].isin(adata_st.var['feature_name'])]
overlap_genes = overlap_genes.to_numpy()
print(overlap_genes.shape)
overlap_genes

(19228,)


array(['FAM87B', 'LINC00115', 'FAM41C', ..., 'MC4R', 'UBOX5-AS1',
       'C20orf203'], dtype=object)

In [11]:
adata_sc.var.index = adata_sc.var['gene']
adata_sc = adata_sc[:, overlap_genes].copy()
adata_sc

AnnData object with n_obs × n_vars = 2601 × 19228
    obs: 'cell', 'cell_type'
    var: 'gene'

In [12]:
unique_genes = ~adata_st.var['feature_name'].duplicated(keep = 'first')
adata_st = adata_st[:, unique_genes].copy()
adata_st

/home/xianjie/.anaconda3/envs/SCSC/lib/python3.11/site-packages/anndata-0.10.7-py3.11.egg/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


AnnData object with n_obs × n_vars = 1600 × 33514
    obs: 'cell', 'cell_type'
    var: 'feature_id', 'feature_name'

In [13]:
adata_st.var.index = adata_st.var['feature_name']
adata_st = adata_st[:, overlap_genes].copy()
adata_st

AnnData object with n_obs × n_vars = 1600 × 19228
    obs: 'cell', 'cell_type'
    var: 'feature_id', 'feature_name'

## Format cells

In [14]:
epi_list = ['Hepatocyte']
df = adata_sc.obs
epi_cells = df.loc[df['cell_type'].isin(epi_list), 'cell'].to_numpy()
epi_cells.shape

(827,)

In [15]:
np.random.seed(123)
selected_cells = np.random.choice(epi_cells, size = n_epi, replace = False)
selected_cells.shape

(700,)

In [16]:
adata_sc = adata_sc[adata_sc.obs['cell'].isin(selected_cells), :].copy()
adata_sc

AnnData object with n_obs × n_vars = 700 × 19228
    obs: 'cell', 'cell_type'
    var: 'gene'

In [17]:
adata_sc.obs['cell_type'] = 'hepatocyte'

In [18]:
df = adata_st.obs
normal_cells = df.loc[df['cell_type'] == 'normal', 'cell'].to_numpy()
tumor_cells = df.loc[df['cell_type'] == 'tumor', 'cell'].to_numpy()
normal_cells.shape, tumor_cells.shape

((500,), (500,))

In [19]:
selected_cells = np.concatenate([
    np.random.choice(normal_cells, size = n_normal, replace = False),
    np.random.choice(tumor_cells, size = n_tumor, replace = False)
])
selected_cells.shape

(300,)

In [20]:
adata_st = adata_st[adata_st.obs['cell'].isin(selected_cells), :].copy()
adata_st

AnnData object with n_obs × n_vars = 300 × 19228
    obs: 'cell', 'cell_type'
    var: 'feature_id', 'feature_name'

## Combine two adata

In [21]:
adata = ad.AnnData(
    obs = pd.concat([adata_sc.obs, adata_st.obs], ignore_index = True),
    var = adata_sc.var[['gene', 'gene']],
    X = np.concatenate([adata_sc.X, adata_st.X], axis = 0)
)
adata

/home/xianjie/.anaconda3/envs/SCSC/lib/python3.11/site-packages/anndata-0.10.7-py3.11.egg/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


AnnData object with n_obs × n_vars = 1000 × 19228
    obs: 'cell', 'cell_type'
    var: 'gene', 'gene'

In [22]:
adata.obs

,cell,cell_type
0,HCC03N_AAACGGGAGATAGGAG,hepatocyte
1,HCC03N_AAACGGGAGGGTTTCT,hepatocyte
2,HCC03N_AAAGATGGTTGTGGCC,hepatocyte
3,HCC03N_AAAGCAACATGCGCAC,hepatocyte
4,HCC03N_AAAGTAGAGCAGGCTA,hepatocyte
...,...,...
995,TTTGGTTCAGACGCTC-1,tumor
996,TTTGGTTGTGCTCTTC-1,tumor
997,TTTGTCAAGGCTAGAC-1,tumor
998,TTTGTCAGTGGGTCAA-1,tumor


In [23]:
adata.var

,gene,gene
gene,,
FAM87B,FAM87B,FAM87B
LINC00115,LINC00115,LINC00115
FAM41C,FAM41C,FAM41C
SAMD11,SAMD11,SAMD11
NOC2L,NOC2L,NOC2L
...,...,...
ODF4,ODF4,ODF4
LINC00670,LINC00670,LINC00670
MC4R,MC4R,MC4R


In [24]:
adata.obs['cell_type'].value_counts()

cell_type
hepatocyte    700
tumor         270
normal         30
Name: count, dtype: int64

In [25]:
save_10x_data(adata, out_dir)